### EXAMPLE OF PROCESSING CKD DATA SET ###

We have two tables:
  * **ckd_enrollment**: Information about members
  * **ckd_claims**: Their claims (these include the combination of inpatient, outpatient and Rx claims)

The data spans the years 2017, 2018 and 2019.

In [1]:
# -----------------------------------------------------------------------------
# INITIALIZATION
# -----------------------------------------------------------------------------
import sys
print(f"Python version: {sys.version}")

import logging
import csv
import gzip
import re
import pandas as pd
import numpy as np
from functools import reduce

import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession
from pyspark import SparkConf
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)


from IPython.core.magic import register_cell_magic

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes=10,
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:

spark = start_spark(num_nodes=10)
#spark.stop()

  
@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)
  



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/03 11:31:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/03 11:31:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/07/03 11:31:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
# -- READ ENROLLMENT AND DATA TABLES
enrollment_file = f"/Users/Charles/DATA/ckd/ckd_enrollment"
logger.info(f">>> Reading enrollment file: {enrollment_file}")
df_enrollment = spark.read.format("parquet").load(enrollment_file)
df_enrollment.createOrReplaceTempView('enrollment')
logger.info(f">>> ENROLLMENT has {df_enrollment.count():,} rows")
logger.info(f">>> ENROLLMENT has {df_enrollment.select('ENROLID').distinct().count():,} unique enrollees")

claims_file = f"/Users/Charles/DATA/ckd/ckd_claims"
logger.info(f">>> Reading claims file: {claims_file}")
df_claims = spark.read.format("parquet").load(claims_file)
df_claims.createOrReplaceTempView('claims')
logger.info(f">>> CLAIMS has {df_claims.count():,} rows")
logger.info(f">>> CLAIMS has {df_claims.select('ENROLID').distinct().count():,} unique enrollees")

# -- NOTE: with the "createOrReplaceTempView" we define a view of these
# -- tables, so we can use them in SQL queries.

2025-07-03 11:32:00,704 INFO     >>> Reading enrollment file: /Users/Charles/DATA/ckd/ckd_enrollment
25/07/03 11:32:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2025-07-03 11:32:02,334 INFO     >>> ENROLLMENT has 2,586,930 rows
2025-07-03 11:32:03,815 INFO     >>> ENROLLMENT has 862,310 unique enrollees    
2025-07-03 11:32:03,816 INFO     >>> Reading claims file: /Users/Charles/DATA/ckd/ckd_claims
2025-07-03 11:32:04,834 INFO     >>> CLAIMS has 253,070,875 rows                
2025-07-03 11:32:12,368 INFO     >>> CLAIMS has 862,310 unique enrollees        


In [ ]:
# Step 1: Count rows per ENROLID
df_counts_per_enrolid = (
    df_enrollment.groupBy("ENROLID")
    .agg(F.count("*").alias("n_rows"))
)

# Step 2: Count how many ENROLIDs have the same number of rows
df_distribution = (
    df_counts_per_enrolid.groupBy("n_rows")
    .agg(F.count("*").alias("n_enrollees"))
    .orderBy("n_rows")
)

df_distribution.show()

df_duplicate_years = (
    df_enrollment
    .groupBy("ENROLID", "YEAR")
    .agg(F.count("*").alias("n_rows"))
    .filter(F.col("n_rows") > 1)
    .orderBy("ENROLID", "YEAR")
)

df_duplicate_years.show()



+------+-----------+
|n_rows|n_enrollees|
+------+-----------+
|     1|         49|
|     2|         47|
|     3|         59|
|     4|         90|
|     5|        100|
|     6|        122|
|     7|        155|
|     8|        170|
|     9|        234|
|    10|        242|
|    11|        304|
|    12|        354|
|    13|        414|
|    14|        463|
|    15|        538|
|    16|        567|
|    17|        601|
|    18|        642|
|    19|        722|
|    20|        746|
+------+-----------+
only showing top 20 rows


+-------+----+------+
|ENROLID|YEAR|n_rows|
+-------+----+------+
+-------+----+------+



### Demographics

In [18]:
df_age_dist = df_enrollment.groupBy("AGE").count().orderBy("AGE")
df_age_pd = df_age_dist.toPandas()
fig = px.bar(
    df_age_pd,
    x="AGE", y="count",
    title="AGE DISTRIBUTION OF CKD ENROLLEES",
    labels={"AGE": "Age", "count": "Number of Enrollees"},
    width=800, height=500
)
fig.update_layout(title_font=dict(size=18), xaxis_tickformat="d")
fig.show()


### Stages of CKD

In [6]:
dx_cols = [col for col in df_claims.columns if col.__contains__("DX")]


# Step 2: Create CKD_STAGE using dynamic matching
def build_ckd_stage_case(dx_cols):
    stage_expr = None
    stage_map = {
        "N181": "CKD Stage 1",
        "N182": "CKD Stage 2",
        "N183": "CKD Stage 3",
        "N184": "CKD Stage 4",
        "N185": "CKD Stage 5",
        "N186": "ESRD",
        "N189": "Unspecified"
    }

    for code, label in stage_map.items():
        # Create a compound OR condition over all DX columns for each code
        condition = reduce(lambda a, b: a | b, [F.col(c).startswith(code) for c in dx_cols])
        if stage_expr is None:
            stage_expr = F.when(condition, label)
        else:
            stage_expr = stage_expr.when(condition, label)
    
    return stage_expr

# Step 3: Filter for CKD codes and extract stage
ckd_filter = reduce(lambda a, b: a | b, [F.col(c).startswith("N18") for c in dx_cols])

df_ckd_claims = (
    df_claims
    .filter(ckd_filter)
    .withColumn("CKD_STAGE", build_ckd_stage_case(dx_cols))
    .filter(F.col("CKD_STAGE").isNotNull())
    .filter(F.col("SVCDATE").isNotNull())
)


### Cost and prevalence by region, geo loc and industry
Run the stages of CKD first then
✅ Count ICD Codes by Industry
✅ Count ICD Codes by Region


In [9]:
import json 
with open("geo_industry_map.json", "r") as f:
    mappings = json.load(f)

region_labels = mappings["region_labels"]
egeoloc_mapping = mappings["egeoloc_labels"]

In [46]:
### df_geoloc_pd is the dataframe grouped by EGEOLOC and ICD10 being N18x 

# Step 1: Aggregate total CKD count by EGEOLOC
df_ckd_state_total = (
    df_geoloc_pd.groupby("EGEOLOC")["count"]
    .sum()
    .reset_index()
)

# Step 2: Map EGEOLOC to state abbreviations using your mapping
df_ckd_state_total["state"] = df_ckd_state_total["EGEOLOC"].map(
    lambda x: egeoloc_mapping.get(x) if isinstance(egeoloc_mapping.get(x), str) and len(egeoloc_mapping.get(x)) == 2 else None
)

# Step 3: Drop rows without valid 2-letter state codes
df_ckd_state_total = df_ckd_state_total.dropna(subset=["state"])

# Step 4: Plot
fig = px.choropleth(
    df_ckd_state_total,
    locations="state",
    locationmode="USA-states",
    color="count",
    color_continuous_scale="Blues",
    scope="usa",
    title="CKD Prevalence by State (Raw Counts)"
)

fig.update_layout(geo=dict(bgcolor="rgba(0,0,0,0)"))
fig.show()


In [27]:
from pyspark.sql import functions as F

# Step 1: Total cost per enrollee per year from claims
df_cost = (
    df_claims
    .groupBy("YEAR", "ENROLID")
    .agg(F.sum("NETPAY").alias("total_cost"))
    .filter("total_cost > 0")  # exclude negatives
)


# Step 2: Get REGION info per enrollee per year
df_region = df_enrollment.select("YEAR", "ENROLID", "INDSTRY").distinct()

# Step 3: Join and group
df_cost_region = (
    df_cost.join(df_region, on=["YEAR", "ENROLID"], how="inner")
    .groupBy("YEAR", "INDSTRY")
    .agg(F.sum("total_cost").alias("industry_total_cost"))
)

# Convert to pandas for plotting
df_cost_region_pd = df_cost_region.toPandas()
import plotly.express as px


df_cost_region_pd["INDSTRY_LABEL"] = df_cost_region_pd["INDSTRY"].astype(str).map(industry_labels)

# Plot
fig = px.bar(
    df_cost_region_pd,
    x="YEAR",
    y="industry_total_cost",
    color="INDSTRY_LABEL",
    barmode="group",
    title="Total Cost per Year by Industry",
    labels={"industry_total_cost": "Total Cost ($)", "YEAR": "Year", "INDSTRYLABEL": "Industry"},
    height=500,
    width=900
)
fig.update_layout(legend_title_text="Industry")
fig.show()

In [69]:
df_person_cost = (
    df_claims
    .groupBy("YEAR", "ENROLID")
    .agg(F.sum("NETPAY").alias("cost"))
    .filter(F.col("cost") > 0)  # Only positive-cost enrollees
)

# Get latest demographic info per enrollee (assuming most recent enrollment year per person)
window_demo = Window.partitionBy("ENROLID").orderBy(F.col("YEAR").desc())

df_enrollee_demo = (
    df_enrollment
    .select("ENROLID", "YEAR", "REGION", "INDSTRY", "EGEOLOC")
    .withColumn("rownum", row_number().over(window_demo))
    .filter(F.col("rownum") == 1)
    .drop("rownum")
)
df_enrollee_demo_clean = df_enrollee_demo.drop("YEAR")

df_cost_demo = (
    df_person_cost
    .join(df_enrollee_demo_clean, on="ENROLID", how="inner")
    .select("ENROLID", "YEAR", "INDSTRY", "REGION", "EGEOLOC", "cost")
)

def compute_avg_cost_by_group(df, group_col):
    return (
        df.groupBy(group_col, "YEAR")
        .agg(
            F.countDistinct("ENROLID").alias("n_enrollees"),
            F.sum("cost").alias("total_cost")
        )
        .withColumn("avg_cost", F.col("total_cost") / F.col("n_enrollees"))
        .orderBy("YEAR", group_col)
    )


df_avg_cost_by_industry = compute_avg_cost_by_group(df_cost_demo, "INDSTRY")
df_avg_cost_by_region = compute_avg_cost_by_group(df_cost_demo, "REGION")
df_avg_cost_by_geoloc = compute_avg_cost_by_group(df_cost_demo, "EGEOLOC")
df_avg_cost_by_industry_pd = df_avg_cost_by_industry.toPandas()
df_avg_cost_by_region_pd = df_avg_cost_by_region.toPandas()
df_avg_cost_by_geoloc_pd = df_avg_cost_by_geoloc.toPandas()


🔶 First heatmap: shows per-capita cost burden across all enrollees in a state. Not CKD-specific costs

🔷 Second heatmap: shows CKD-specific cost per CKD patient, which tends to be higher, especially in places like AK with few CKD enrollees but high costs per person.



In [77]:
# First heatmap: shows per-capita cost burden across all enrollees in a state.

# Compute 3-year average cost per enrollee by EGEOLOC from df_avg_cost_by_geoloc_pd
df_avg_cost_state = (
    df_avg_cost_by_geoloc_pd
    .groupby("EGEOLOC")
    .agg(
        total_cost=("total_cost", "sum"),
        total_n=("n_enrollees", "sum")
    )
    .assign(avg_cost=lambda df: df["total_cost"] / df["total_n"])
    .reset_index()
)

# Map EGEOLOC to 2-letter state codes
df_avg_cost_state["state"] = df_avg_cost_state["EGEOLOC"].map(
    lambda x: egeoloc_mapping.get(str(x)) if isinstance(egeoloc_mapping.get(str(x)), str) and len(egeoloc_mapping.get(str(x))) == 2 else None
)

# Drop rows with no valid state abbreviation
df_avg_cost_state = df_avg_cost_state.dropna(subset=["state"])
import plotly.express as px

fig = px.choropleth(
    df_avg_cost_state,
    locations="state",
    locationmode="USA-states",
    color="avg_cost",
    color_continuous_scale="OrRd",
    scope="usa",
    title="Cost per Enrollee (3-Year Average over all enrollees) by State"
)
fig.update_layout(geo=dict(bgcolor="rgba(0,0,0,0)"))
fig.show()


In [73]:
df_avg_cost_by_geoloc_pd.show(3)


+-------+----+-------+------+-------+-----------------+
|ENROLID|YEAR|INDSTRY|REGION|EGEOLOC|             cost|
+-------+----+-------+------+-------+-----------------+
| 166103|2017|      7|     2|     18|          5749.49|
| 166103|2018|      7|     2|     18|           2422.5|
| 166103|2019|      7|     2|     18|6119.379999999999|
+-------+----+-------+------+-------+-----------------+
only showing top 3 rows


+-------+----+-------+------+-------+-----------------+
|ENROLID|YEAR|INDSTRY|REGION|EGEOLOC|             cost|
+-------+----+-------+------+-------+-----------------+
| 166103|2017|      7|     2|     18|          5749.49|
| 166103|2018|      7|     2|     18|           2422.5|
| 166103|2019|      7|     2|     18|6119.379999999999|
+-------+----+-------+------+-------+-----------------+
only showing top 3 rows


In [75]:
# Second heatmap: 
df_cost_demo_pd = df_cost_demo.toPandas()

df_avg_cost_state = (
    df_cost_demo_pd
    .groupby("EGEOLOC")
    .agg(
        total_cost=("cost", "sum"),
        n_enrollees=("ENROLID", "nunique")
    )
    .assign(avg_cost=lambda df: df["total_cost"] / df["n_enrollees"])
    .reset_index()
)

# Step 2: Map EGEOLOC to 2-letter state abbreviation
df_avg_cost_state["state"] = df_avg_cost_state["EGEOLOC"].map(
    lambda x: egeoloc_mapping.get(x) if isinstance(egeoloc_mapping.get(x), str) and len(egeoloc_mapping.get(x)) == 2 else None
)

# Step 3: Drop unmapped
df_avg_cost_state = df_avg_cost_state.dropna(subset=["state"])

df_avg_cost_state.query("EGEOLOC == '61'")

,EGEOLOC,total_cost,n_enrollees,avg_cost,state
47,61,43136766.02,400,107841.91505,AK


In [79]:
# Join cost data with CKD enrollees
df_ckd_cost_demo_pd = (
    df_cost_demo.join(df_ckd_enrollees, on="ENROLID", how="inner")
).toPandas()

df_avg_ckd_cost_state = (
    df_ckd_cost_demo_pd
    .groupby("EGEOLOC")
    .agg(
        total_cost=("cost", "sum"),
        n_enrollees=("ENROLID", "nunique")
    )
    .assign(avg_cost=lambda df: df["total_cost"] / df["n_enrollees"])
    .reset_index()
)
df_avg_ckd_cost_state["state"] = df_avg_ckd_cost_state["EGEOLOC"].map(
    lambda x: egeoloc_mapping.get(str(x)) if isinstance(egeoloc_mapping.get(str(x)), str) and len(egeoloc_mapping.get(str(x))) == 2 else None
)
df_avg_ckd_cost_state = df_avg_ckd_cost_state.dropna(subset=["state"])

fig = px.choropleth(
    df_avg_ckd_cost_state,
    locations="state",
    locationmode="USA-states",
    color="avg_cost",
    color_continuous_scale="OrRd",
    scope="usa",
    title="Cost per Enrollee with CKD Claims only (3-Year Average) by State"
)
fig.update_layout(geo=dict(bgcolor="rgba(0,0,0,0)"))
fig.show()
